In [ ]:
from collections import deque
import heapq
import time
import math

In [ ]:
# Initial state
#
# (bearing_inspected,
#  bearing_worn_confirmed,
#  bearing_replaced,
#  lubrication_checked,
#  vibration_tested,
#  temperature_tested,
#  pump_tested,
#  pump_operational)

initial_state = (
    False,  # bearing_inspected
    False,  # bearing_worn_confirmed
    False,  # bearing_replaced
    False,  # lubrication_checked
    False,  # vibration_tested
    False,  # temperature_tested
    False,  # pump_tested
    False   # pump_operational
)

# Goal state
goal_state = (
    True,   # bearing_inspected
    True,   # bearing_worn_confirmed
    True,   # bearing_replaced
    False,  # lubrication_checked - not required
    True,   # vibration_tested
    True,   # temperature_tested
    True,   # pump_tested
    True    # pump_operational
)

print("Initial State:", initial_state)
print("Goal State:", goal_state)

Initial State: (False, False, False, False, False, False, False, False)
Goal State: (True, True, True, False, True, True, True, True)


In [ ]:
ACTIONS = {
    "Inspect Bearing": {
        "cost": 1
    },

    "Confirm Bearing Wear": {
        "cost": 1
    },

    "Check Lubrication": {
        "cost": 1
    },

    "Lubricate Bearing": {
        "cost": 2
    },

    "Replace Bearing": {
        "cost": 5
    },

    "Run Vibration Test": {
        "cost": 2
    },

    "Run Temperature Test": {
        "cost": 2
    },

    "Test Pump": {
        "cost": 2
    },

    "Restore Pump Operation": {
        "cost": 1
    }
}

print("Available actions:")
for action, data in ACTIONS.items():
    print(f"- {action}: cost = {data['cost']}")

Available actions:
- Inspect Bearing: cost = 1
- Confirm Bearing Wear: cost = 1
- Check Lubrication: cost = 1
- Lubricate Bearing: cost = 2
- Replace Bearing: cost = 5
- Run Vibration Test: cost = 2
- Run Temperature Test: cost = 2
- Test Pump: cost = 2
- Restore Pump Operation: cost = 1


In [ ]:
def is_valid_action(state, action):
    """
    Determines whether an action can legally be performed
    in the current state.
    """

    bearing_inspected = state[0]
    bearing_worn_confirmed = state[1]
    bearing_replaced = state[2]
    lubrication_checked = state[3]
    vibration_tested = state[4]
    temperature_tested = state[5]
    pump_tested = state[6]
    pump_operational = state[7]

    # Inspect Bearing
    if action == "Inspect Bearing":
        return not bearing_inspected

    # Confirm Bearing Wear
    elif action == "Confirm Bearing Wear":
        return bearing_inspected and not bearing_worn_confirmed

    # Check Lubrication
    elif action == "Check Lubrication":
        return bearing_inspected and not lubrication_checked

    # Lubricate Bearing
    elif action == "Lubricate Bearing":
        return lubrication_checked and not bearing_replaced

    # Replace Bearing
    elif action == "Replace Bearing":
        return bearing_worn_confirmed and not bearing_replaced

    # Vibration test
    elif action == "Run Vibration Test":
        return bearing_replaced and not vibration_tested

    # Temperature test
    elif action == "Run Temperature Test":
        return bearing_replaced and not temperature_tested

    # Pump test
    elif action == "Test Pump":
        return (
            bearing_replaced
            and vibration_tested
            and temperature_tested
            and not pump_tested
        )

    # Restore operation
    elif action == "Restore Pump Operation":
        return pump_tested and not pump_operational

    return False

In [ ]:
def apply_action(state, action):
    """
    Applies an action to a state and returns the new state.
    Assumes the action is valid.
    """

    new_state = list(state)

    if action == "Inspect Bearing":
        new_state[0] = True

    elif action == "Confirm Bearing Wear":
        new_state[1] = True

    elif action == "Check Lubrication":
        new_state[3] = True

    elif action == "Lubricate Bearing":
        pass
        # Lubrication does not directly affect the main repair path.

    elif action == "Replace Bearing":
        new_state[2] = True

    elif action == "Run Vibration Test":
        new_state[4] = True

    elif action == "Run Temperature Test":
        new_state[5] = True

    elif action == "Test Pump":
        new_state[6] = True

    elif action == "Restore Pump Operation":
        new_state[7] = True

    return tuple(new_state)

In [ ]:
def get_neighbors(state):
    """
    Returns all valid next states from the current state.

    Each result contains:
    (new_state, action, action_cost)
    """

    neighbors = []

    for action, data in ACTIONS.items():

        if is_valid_action(state, action):

            new_state = apply_action(state, action)

            neighbors.append(
                (
                    new_state,
                    action,
                    data["cost"]
                )
            )

    return neighbors

In [ ]:
print("Initial State:")
print(initial_state)

print("\nValid actions from initial state:")

for state, action, cost in get_neighbors(initial_state):
    print(f"{action} | Cost: {cost}")

Initial State:
(False, False, False, False, False, False, False, False)

Valid actions from initial state:
Inspect Bearing | Cost: 1


In [ ]:
state = initial_state

manual_actions = [
    "Inspect Bearing",
    "Confirm Bearing Wear",
    "Replace Bearing",
    "Run Vibration Test",
    "Run Temperature Test",
    "Test Pump",
    "Restore Pump Operation"
]

print("Starting state:")
print(state)

for action in manual_actions:

    if not is_valid_action(state, action):
        print(f"\nINVALID ACTION: {action}")
        break

    state = apply_action(state, action)

    print(f"\nAction: {action}")
    print(f"New state: {state}")

print("\nGoal reached:", state == goal_state)

Starting state:
(False, False, False, False, False, False, False, False)

Action: Inspect Bearing
New state: (True, False, False, False, False, False, False, False)

Action: Confirm Bearing Wear
New state: (True, True, False, False, False, False, False, False)

Action: Replace Bearing
New state: (True, True, True, False, False, False, False, False)

Action: Run Vibration Test
New state: (True, True, True, False, True, False, False, False)

Action: Run Temperature Test
New state: (True, True, True, False, True, True, False, False)

Action: Test Pump
New state: (True, True, True, False, True, True, True, False)

Action: Restore Pump Operation
New state: (True, True, True, False, True, True, True, True)

Goal reached: True


In [ ]:
def is_goal(state):
    """
    Checks whether the required goal conditions are satisfied.
    """

    return (
        state[0] and  # bearing inspected
        state[1] and  # bearing wear confirmed
        state[2] and  # bearing replaced
        state[4] and  # vibration tested
        state[5] and  # temperature tested
        state[6] and  # pump tested
        state[7]      # pump operational
    )

In [ ]:
def reconstruct_path(parent, action_taken, goal):
    """
    Reconstructs the sequence of states and actions
    from the initial state to the goal.
    """

    states = []
    actions = []

    current = goal

    while current is not None:

        states.append(current)

        if current in action_taken:
            actions.append(action_taken[current])

        current = parent[current]

    states.reverse()
    actions.reverse()

    return states, actions

In [ ]:
def bfs(initial_state):
    start_time = time.perf_counter()

    queue = deque([initial_state])

    visited = {initial_state}

    parent = {
        initial_state: None
    }

    action_taken = {}

    nodes_expanded = 0

    while queue:

        current = queue.popleft()

        nodes_expanded += 1

        # Goal check
        if is_goal(current):

            states, actions = reconstruct_path(
                parent,
                action_taken,
                current
            )

            execution_time = time.perf_counter() - start_time

            return {
                "found": True,
                "states": states,
                "actions": actions,
                "path_cost": sum(
                    ACTIONS[action]["cost"]
                    for action in actions
                ),
                "nodes_expanded": nodes_expanded,
                "execution_time": execution_time
            }

        # Explore neighbors
        for next_state, action, cost in get_neighbors(current):

            if next_state not in visited:

                visited.add(next_state)

                parent[next_state] = current

                action_taken[next_state] = action

                queue.append(next_state)

    execution_time = time.perf_counter() - start_time

    return {
        "found": False,
        "states": [],
        "actions": [],
        "path_cost": math.inf,
        "nodes_expanded": nodes_expanded,
        "execution_time": execution_time
    }

In [ ]:
bfs_result = bfs(initial_state)

print("BFS RESULT")
print("=" * 40)

print("Solution found:", bfs_result["found"])

print("\nMaintenance path:")

for i, action in enumerate(bfs_result["actions"], start=1):
    print(f"{i}. {action}")

print("\nPath cost:", bfs_result["path_cost"])
print("Nodes expanded:", bfs_result["nodes_expanded"])
print("Execution time:", bfs_result["execution_time"])

BFS RESULT
Solution found: True

Maintenance path:
1. Inspect Bearing
2. Confirm Bearing Wear
3. Replace Bearing
4. Run Vibration Test
5. Run Temperature Test
6. Test Pump
7. Restore Pump Operation

Path cost: 14
Nodes expanded: 16
Execution time: 0.00017024400005993812


In [ ]:
def dfs(initial_state):
    start_time = time.perf_counter()

    stack = [initial_state]

    visited = {initial_state}

    parent = {
        initial_state: None
    }

    action_taken = {}

    nodes_expanded = 0

    while stack:

        current = stack.pop()

        nodes_expanded += 1

        # Goal check
        if is_goal(current):

            states, actions = reconstruct_path(
                parent,
                action_taken,
                current
            )

            execution_time = time.perf_counter() - start_time

            return {
                "found": True,
                "states": states,
                "actions": actions,
                "path_cost": sum(
                    ACTIONS[action]["cost"]
                    for action in actions
                ),
                "nodes_expanded": nodes_expanded,
                "execution_time": execution_time
            }

        # Explore neighbors
        for next_state, action, cost in get_neighbors(current):

            if next_state not in visited:

                visited.add(next_state)

                parent[next_state] = current

                action_taken[next_state] = action

                stack.append(next_state)

    execution_time = time.perf_counter() - start_time

    return {
        "found": False,
        "states": [],
        "actions": [],
        "path_cost": math.inf,
        "nodes_expanded": nodes_expanded,
        "execution_time": execution_time
    }

In [ ]:
dfs_result = dfs(initial_state)

print("DFS RESULT")
print("=" * 40)

print("Solution found:", dfs_result["found"])

print("\nMaintenance path:")

for i, action in enumerate(dfs_result["actions"], start=1):
    print(f"{i}. {action}")

print("\nPath cost:", dfs_result["path_cost"])
print("Nodes expanded:", dfs_result["nodes_expanded"])
print("Execution time:", dfs_result["execution_time"])

DFS RESULT
Solution found: True

Maintenance path:
1. Inspect Bearing
2. Check Lubrication
3. Confirm Bearing Wear
4. Replace Bearing
5. Run Temperature Test
6. Run Vibration Test
7. Test Pump
8. Restore Pump Operation

Path cost: 15
Nodes expanded: 9
Execution time: 8.21820000282969e-05


In [ ]:
def heuristic(state):
    """
    Estimated remaining maintenance cost.

    This heuristic assumes the required remaining
    actions must still be performed.
    """

    cost = 0

    bearing_inspected = state[0]
    bearing_worn_confirmed = state[1]
    bearing_replaced = state[2]
    vibration_tested = state[4]
    temperature_tested = state[5]
    pump_tested = state[6]
    pump_operational = state[7]

    if not bearing_inspected:
        cost += ACTIONS["Inspect Bearing"]["cost"]

    if not bearing_worn_confirmed:
        cost += ACTIONS["Confirm Bearing Wear"]["cost"]

    if not bearing_replaced:
        cost += ACTIONS["Replace Bearing"]["cost"]

    if not vibration_tested:
        cost += ACTIONS["Run Vibration Test"]["cost"]

    if not temperature_tested:
        cost += ACTIONS["Run Temperature Test"]["cost"]

    if not pump_tested:
        cost += ACTIONS["Test Pump"]["cost"]

    if not pump_operational:
        cost += ACTIONS["Restore Pump Operation"]["cost"]

    return cost

In [ ]:
print("Heuristic at initial state:")
print(heuristic(initial_state))

print("\nHeuristic at goal:")
print(heuristic(goal_state))

Heuristic at initial state:
14

Heuristic at goal:
0


In [ ]:
def astar(initial_state):
    start_time = time.perf_counter()

    # Priority queue entries:
    # (f_score, counter, state)

    counter = 0

    initial_f = heuristic(initial_state)

    priority_queue = [
        (initial_f, counter, initial_state)
    ]

    # Cheapest known cost to each state
    g_cost = {
        initial_state: 0
    }

    parent = {
        initial_state: None
    }

    action_taken = {}

    visited = set()

    nodes_expanded = 0

    while priority_queue:

        f_score, _, current = heapq.heappop(priority_queue)

        # Skip if already processed
        if current in visited:
            continue

        visited.add(current)

        nodes_expanded += 1

        # Goal check
        if is_goal(current):

            states, actions = reconstruct_path(
                parent,
                action_taken,
                current
            )

            execution_time = time.perf_counter() - start_time

            return {
                "found": True,
                "states": states,
                "actions": actions,
                "path_cost": g_cost[current],
                "nodes_expanded": nodes_expanded,
                "execution_time": execution_time
            }

        # Explore neighbors
        for next_state, action, action_cost in get_neighbors(current):

            new_g_cost = (
                g_cost[current]
                + action_cost
            )

            # If this is a better route
            if (
                next_state not in g_cost
                or new_g_cost < g_cost[next_state]
            ):

                g_cost[next_state] = new_g_cost

                parent[next_state] = current

                action_taken[next_state] = action

                counter += 1

                f_score = (
                    new_g_cost
                    + heuristic(next_state)
                )

                heapq.heappush(
                    priority_queue,
                    (
                        f_score,
                        counter,
                        next_state
                    )
                )

    execution_time = time.perf_counter() - start_time

    return {
        "found": False,
        "states": [],
        "actions": [],
        "path_cost": math.inf,
        "nodes_expanded": nodes_expanded,
        "execution_time": execution_time
    }

In [ ]:
astar_result = astar(initial_state)

print("A* RESULT")
print("=" * 40)

print("Solution found:", astar_result["found"])

print("\nMaintenance path:")

for i, action in enumerate(astar_result["actions"], start=1):
    print(f"{i}. {action}")

print("\nPath cost:", astar_result["path_cost"])
print("Nodes expanded:", astar_result["nodes_expanded"])
print("Execution time:", astar_result["execution_time"])

A* RESULT
Solution found: True

Maintenance path:
1. Inspect Bearing
2. Confirm Bearing Wear
3. Replace Bearing
4. Run Vibration Test
5. Run Temperature Test
6. Test Pump
7. Restore Pump Operation

Path cost: 14
Nodes expanded: 9
Execution time: 9.38339999265736e-05


In [ ]:
results = {
    "BFS": bfs_result,
    "DFS": dfs_result,
    "A*": astar_result
}

print("=" * 70)
print("SEARCH ALGORITHM COMPARISON")
print("=" * 70)

for algorithm, result in results.items():

    print(f"\n{algorithm}")
    print("-" * 40)

    print("Solution found:", result["found"])
    print("Actions:", len(result["actions"]))
    print("Path cost:", result["path_cost"])
    print("Nodes expanded:", result["nodes_expanded"])
    print(
        "Execution time:",
        f"{result['execution_time']:.8f}",
        "seconds"
    )

SEARCH ALGORITHM COMPARISON

BFS
----------------------------------------
Solution found: True
Actions: 7
Path cost: 14
Nodes expanded: 16
Execution time: 0.00017024 seconds

DFS
----------------------------------------
Solution found: True
Actions: 8
Path cost: 15
Nodes expanded: 9
Execution time: 0.00008218 seconds

A*
----------------------------------------
Solution found: True
Actions: 7
Path cost: 14
Nodes expanded: 9
Execution time: 0.00009383 seconds


In [ ]:
import pandas as pd

comparison = []

for algorithm, result in results.items():

    comparison.append({
        "Algorithm": algorithm,
        "Solution Found": result["found"],
        "Number of Actions": len(result["actions"]),
        "Path Cost": result["path_cost"],
        "Nodes Expanded": result["nodes_expanded"],
        "Execution Time (sec)": result["execution_time"]
    })

comparison_df = pd.DataFrame(comparison)

comparison_df

,Algorithm,Solution Found,Number of Actions,Path Cost,Nodes Expanded,Execution Time (sec)
0,BFS,True,7,14,16,0.000170
1,DFS,True,8,15,9,0.000082
2,A*,True,7,14,9,0.000094


In [ ]:
for algorithm, result in results.items():

    print("\n" + "=" * 60)
    print(algorithm)
    print("=" * 60)

    for i, action in enumerate(result["actions"], start=1):
        print(f"{i}. {action}")


BFS
1. Inspect Bearing
2. Confirm Bearing Wear
3. Replace Bearing
4. Run Vibration Test
5. Run Temperature Test
6. Test Pump
7. Restore Pump Operation

DFS
1. Inspect Bearing
2. Check Lubrication
3. Confirm Bearing Wear
4. Replace Bearing
5. Run Temperature Test
6. Run Vibration Test
7. Test Pump
8. Restore Pump Operation

A*
1. Inspect Bearing
2. Confirm Bearing Wear
3. Replace Bearing
4. Run Vibration Test
5. Run Temperature Test
6. Test Pump
7. Restore Pump Operation


In [ ]:
def validate_path(initial_state, actions):
    state = initial_state

    for action in actions:

        if not is_valid_action(state, action):
            return False, f"Invalid action: {action}"

        state = apply_action(state, action)

    if not is_goal(state):
        return False, "Path does not reach goal."

    return True, "Valid path"


for algorithm, result in results.items():

    valid, message = validate_path(
        initial_state,
        result["actions"]
    )

    print(f"{algorithm}: {message}")

BFS: Valid path
DFS: Valid path
A*: Valid path


In [ ]:
best_algorithm = "A*"

best_result = results[best_algorithm]

print("\n" + "=" * 60)
print("RECOMMENDED MAINTENANCE PLAN")
print("=" * 60)

print("\nEquipment: P-101")
print("Fault: Suspected Bearing Wear")
print(f"Planning Algorithm: {best_algorithm}")

print("\nMaintenance Sequence:")

for i, action in enumerate(best_result["actions"], start=1):
    print(f"  {i}. {action}")

print("\n" + "-" * 60)

print("Total Actions:",
      len(best_result["actions"]))

print("Total Maintenance Cost:",
      best_result["path_cost"])

print("States Explored:",
      best_result["nodes_expanded"])

print(
    "Planning Time:",
    f"{best_result['execution_time']:.8f}",
    "seconds"
)


RECOMMENDED MAINTENANCE PLAN

Equipment: P-101
Fault: Suspected Bearing Wear
Planning Algorithm: A*

Maintenance Sequence:
  1. Inspect Bearing
  2. Confirm Bearing Wear
  3. Replace Bearing
  4. Run Vibration Test
  5. Run Temperature Test
  6. Test Pump
  7. Restore Pump Operation

------------------------------------------------------------
Total Actions: 7
Total Maintenance Cost: 14
States Explored: 9
Planning Time: 0.00009383 seconds


In [ ]:
SCENARIOS = {

    "Bearing Failure - P-101": {
        "equipment": "P-101",
        "fault": "Bearing Wear",
        "initial_state": initial_state
    }

}

for name, scenario in SCENARIOS.items():

    print(name)
    print("Equipment:", scenario["equipment"])
    print("Fault:", scenario["fault"])

Bearing Failure - P-101
Equipment: P-101
Fault: Bearing Wear


In [ ]:
def run_all_algorithms(initial_state):

    bfs_result = bfs(initial_state)

    dfs_result = dfs(initial_state)

    astar_result = astar(initial_state)

    return {
        "BFS": bfs_result,
        "DFS": dfs_result,
        "A*": astar_result
    }


experiment_results = run_all_algorithms(initial_state)

for algorithm, result in experiment_results.items():

    print(
        algorithm,
        "→",
        "Found" if result["found"] else "Not Found",
        "| Cost:",
        result["path_cost"],
        "| Nodes:",
        result["nodes_expanded"]
    )

BFS → Found | Cost: 14 | Nodes: 16
DFS → Found | Cost: 15 | Nodes: 9
A* → Found | Cost: 14 | Nodes: 9
